In [1]:
import h5py
import numpy as np
import numpy.lib.recfunctions as rfn
from scipy.spatial.distance import pdist
from scipy.spatial.distance import squareform

from scipy.spatial.distance import cdist

from sklearn.cluster import DBSCAN

import boost_histogram as bh
import hist

In [86]:
f = h5py.File('PicoRun6.1c_1E17_RHC.flow.0000000.FLOW.hdf5')
# f = h5py.File('packet-0050015-2024_07_08_13_37_49_CDT.FLOW.hdf5')

In [87]:
f.keys()

<KeysViewHDF5 ['charge', 'combined', 'geometry_info', 'lar_info', 'light', 'mc_truth', 'run_info']>

In [102]:
f['/charge/calib_prompt_hits/ref/mc_truth/calib_prompt_hit_backtrack/ref_region'][0]

(0, 1)

In [99]:
print(f['/mc_truth/calib_prompt_hit_backtrack/data'].dtype)
f['/mc_truth/calib_prompt_hit_backtrack/data'][0]

[('event_ids', '<i8', (1,)), ('segment_ids', '<i8', (20,)), ('fraction', '<f8', (20,)), ('file_traj_ids', '<i8', (20,)), ('fraction_traj', '<f8', (20,))]


([1], [406, 407, 408, 405,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1], [0.43582198, 0.30597213, 0.25798021, 0.17722989, 0.        , 0.        , 0.        , 0.        , 0.        , 0.        , 0.        , 0.        , 0.        , 0.        , 0.        , 0.        , 0.        , 0.        , 0.        , 0.        ], [388,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1], [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [64]:
hits = f['/charge/calib_prompt_hits/data'][:5000]

In [65]:
def group_hits_by_pxl(hits):
    tol = 1E-4 # cm
    points_yz = np.column_stack([hits['io_group'], hits['io_channel'], hits['y'], hits['z']])
    clustering = DBSCAN(eps=tol, min_samples=1).fit(points_yz)
    components = clustering.components_
    labels = clustering.labels_
    # isconsistent = points_yz[
    norm = np.linalg.norm(points_yz-components, axis=1)
    # extended_hits = rfn.append_fields(hits, names=['new_io_group', 'new_io_channel', 
    #                                               'new_y', 'fit_z'
    #                                               ], data=[totQ_cp, totN_cp], usemask=False)
    return labels, norm>tol

In [66]:
labels, masks = group_hits_by_pxl(hits)
extended_hits = rfn.append_fields(hits, names=['label', 'mask'], data=[labels, masks], dtypes=[np.int_, np.bool_], usemask=False)

In [67]:
sel = np.logical_not(extended_hits['mask'])
sel_hits = extended_hits[sel]

In [68]:
distance = pdist(extended_hits[sel]['ts_pps'].reshape(-1, 1))

In [76]:
h = (
    hist.Hist.new.Reg(100, -0.5, 99.5, name="x", label="dt [0.1us]")
    .Double()
)

In [77]:
unique_pxls = np.unique(sel_hits['label'])
for pxl in unique_pxls:
    m = sel_hits['label'] == pxl
    distance = pdist(sel_hits['ts_pps'][m].reshape(-1, 1))
    h.fill(distance)

In [78]:
h

Hist(Regular(100, -0.5, 99.5, name='x', label='dt [0.1us]'), storage=Double()) # Sum: 18.0 (430.0 with flow)

In [79]:
np.where(h.view()>0)[0]

array([28, 29, 33, 57, 66])

In [80]:
h.axes[0].centers[np.where(h.view()>0)[0]]

array([28., 29., 33., 57., 66.])